In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"

HARDCODE_TEST_ID = [1629027, 202696]
RANDOMSEED = 404
np.random.seed(RANDOMSEED) 

In [8]:
player_path = DATA_DIR / "cohort_1999_2019.csv"
player_df = pd.read_csv(player_path)
career_path = DATA_DIR / "career_totals_targets.csv"
career_total_df = pd.read_csv(career_path)
game_path = DATA_DIR / "raw_game_logs_S1_to_S4.csv"
raw_games_df = pd.read_csv(game_path)
cluster_pather = PROJECT_ROOT / "kmeans_k5_outputs_split" / "cluster_modeling_predictors_k5.csv"
cluster_df = pd.read_csv(cluster_pather)

In [9]:
# ---------------------------------------------------------
# 1. Count total players in the main dataset
# ---------------------------------------------------------
total_players = player_df['PERSON_ID'].nunique()
print(f"Total unique players in player_df: {total_players}")

# ---------------------------------------------------------
# 2. Verify players in other datasets exist in the main dataset
# ---------------------------------------------------------
# Extract unique sets of player IDs
main_players = set(player_df['PERSON_ID'])
career_players = set(career_total_df['PLAYER_ID'])
game_players = set(raw_games_df['Player_ID'])
cluster_players = set(cluster_df['Player_ID'])

# Check for subset conditions
career_in_main = career_players.issubset(main_players)
games_in_main = game_players.issubset(main_players)
cluster_in_main = cluster_players.issubset(main_players)

print("\n--- Verification: Other datasets matching player_df ---")
print(f"All career_total_df players in player_df: {career_in_main}")
if not career_in_main:
    print(f"  -> Found {len(career_players - main_players)} players in career_total_df NOT in player_df.")

print(f"All raw_games_df players in player_df: {games_in_main}")
if not games_in_main:
    print(f"  -> Found {len(game_players - main_players)} players in raw_games_df NOT in player_df.")

print(f"All cluster_df players in player_df: {cluster_in_main}")
if not cluster_in_main:
    print(f"  -> Found {len(cluster_players - main_players)} players in cluster_df NOT in player_df.")

# ---------------------------------------------------------
# 3. Verify cluster_df properties (Unique rows & Full Assignment)
# ---------------------------------------------------------
print("\n--- Verification: cluster_df structure ---")

# Check if there is exactly 1 unique row for each player
is_unique = cluster_df['Player_ID'].is_unique
print(f"1 unique row for each player in cluster_df: {is_unique}")
if not is_unique:
    duplicate_count = cluster_df.duplicated('Player_ID').sum()
    print(f"  -> Warning: Found {duplicate_count} duplicate player records in cluster_df.")

# Verify that ALL players from the main player_df have been assigned a cluster
main_in_cluster = main_players.issubset(cluster_players)
print(f"All player_df players are assigned a cluster in cluster_df: {main_in_cluster}")
if not main_in_cluster:
    print(f"  -> Warning: {len(main_players - cluster_players)} players from player_df are missing from cluster_df.")

# ---------------------------------------------------------
# 4. Compare player lists across cluster, games, and career
# ---------------------------------------------------------
# Extract unique sets of player IDs (if not already done)
career_players = set(career_total_df['PLAYER_ID'])
game_players = set(raw_games_df['Player_ID'])
cluster_players = set(cluster_df['Player_ID'])

print("\n--- Verification: Cross-checking cluster, games, and career datasets ---")

# Check if all players in cluster_df are in raw_games_df
cluster_in_games = cluster_players.issubset(game_players)
print(f"All cluster_df players exist in raw_games_df: {cluster_in_games}")
if not cluster_in_games:
    print(f"  -> Found {len(cluster_players - game_players)} players in cluster_df NOT in raw_games_df.")

# Check if all players in cluster_df are in career_total_df
cluster_in_career = cluster_players.issubset(career_players)
print(f"All cluster_df players exist in career_total_df: {cluster_in_career}")
if not cluster_in_career:
    print(f"  -> Found {len(cluster_players - career_players)} players in cluster_df NOT in career_total_df.")

# Check if all three datasets have the exact same list of players
are_lists_identical = (cluster_players == game_players == career_players)
print(f"\nAre the unique player lists for all three dataframes EXACTLY the same?: {are_lists_identical}")

# If they aren't identical, let's see which one has the most players to help you pick a baseline
if not are_lists_identical:
    print("\n--- Breakdown of unique player counts ---")
    print(f"raw_games_df players:    {len(game_players)}")
    print(f"career_total_df players: {len(career_players)}")
    print(f"cluster_df players:      {len(cluster_players)}")
    
    # Identify overlaps / missing pieces to decide on the baseline
    missing_from_cluster = (game_players | career_players) - cluster_players
    print(f"\nTotal unique players across games and career NOT in cluster_df: {len(missing_from_cluster)}")

Total unique players in player_df: 1247

--- Verification: Other datasets matching player_df ---
All career_total_df players in player_df: True
All raw_games_df players in player_df: True
All cluster_df players in player_df: True

--- Verification: cluster_df structure ---
1 unique row for each player in cluster_df: True
All player_df players are assigned a cluster in cluster_df: False
  -> Warning: 189 players from player_df are missing from cluster_df.

--- Verification: Cross-checking cluster, games, and career datasets ---
All cluster_df players exist in raw_games_df: True
All cluster_df players exist in career_total_df: False
  -> Found 29 players in cluster_df NOT in career_total_df.

Are the unique player lists for all three dataframes EXACTLY the same?: False

--- Breakdown of unique player counts ---
raw_games_df players:    1058
career_total_df players: 1049
cluster_df players:      1058

Total unique players across games and career NOT in cluster_df: 20


# Train Test Split

We split unique players in career_total_df 80/20 with the test set players having AT LEAST 7 SEASONS of data

In [16]:
# 0. Pre-process seasons: handle duplicates (multiple teams) and sort chronologically
player_seasons = career_total_df[['PLAYER_ID', 'SEASON_ID']].copy()
player_seasons['START_YEAR'] = player_seasons['SEASON_ID'].str.split('-').str[0].astype(int)

# Drop duplicate seasons for players who played on multiple teams in the same year
player_seasons = player_seasons.drop_duplicates(subset=['PLAYER_ID', 'START_YEAR'])
player_seasons = player_seasons.sort_values(['PLAYER_ID', 'START_YEAR'])

# Group into lists of unique, sorted starting years
grouped_seasons = player_seasons.groupby('PLAYER_ID')['START_YEAR'].apply(list)

# 1. Get the unique season counts for each player
season_counts = grouped_seasons.apply(len)
players_gte_7 = season_counts[season_counts >= 7].index
total_eligible_players = len(players_gte_7)
test_set_size = int(total_eligible_players * 0.3)

# 2. Verify and isolate hardcoded IDs
valid_hardcode_ids = [pid for pid in HARDCODE_TEST_ID if pid in season_counts.index]
remaining_test_spots = test_set_size - len(valid_hardcode_ids)

# 3. Identify eligible players (>= 7 seasons) and remove hardcoded ones from the pool
eligible_test_players = season_counts[season_counts.between(7, 10)].index
eligible_pool = [pid for pid in eligible_test_players if pid not in valid_hardcode_ids]

# 4. Randomly sample the remaining required test players
if len(eligible_pool) < remaining_test_spots:
    sampled_test_players = eligible_pool
else:
    sampled_test_players = np.random.choice(eligible_pool, size=remaining_test_spots, replace=False).tolist()

final_test_players = valid_hardcode_ids + sampled_test_players

# 5. Extract unique Player_ID to Player_Name mapping from career_total_df
player_name_map = career_total_df[['PLAYER_ID', 'PLAYER_NAME']].drop_duplicates().set_index('PLAYER_ID')['PLAYER_NAME']

# -------------------------------------------------------------------------
# NEW LOGIC: Verify Consecutive Seasons and Generate Sliding Windows
# -------------------------------------------------------------------------

test_set = set(final_test_players)
players_lt_7 = set(season_counts[season_counts < 7].index)

samples = []

for player_id, start_years in grouped_seasons.items():
    # Skip players with < 7 unique seasons entirely
    if player_id in players_lt_7 or len(start_years) < 7:
        continue
        
    split = 'Test' if player_id in test_set else 'Train'
    player_name = player_name_map.get(player_id, "Unknown")
    
    # Total possible windows of length 7
    num_windows = len(start_years) - 6
    
    if split == 'Test':
        # TEST: Find the FIRST valid 7-consecutive-season window
        for i in range(num_windows):
            # If the 7th year is exactly 6 years after the 1st year, the span is consecutive
            if start_years[i + 6] == start_years[i] + 6:
                base_year = start_years[i]
                samples.append({
                    'SPLIT': 'Test',
                    'PLAYER_NAME': player_name,
                    'PLAYER_ID': player_id,
                    'INPUT_SEASON': f"{base_year}-{base_year + 4}",
                    'FORECAST_SEASON': f"{base_year + 4}-{base_year + 7}"
                })
                break # Stop after 1 sample for test
                
    elif split == 'Train':
        # TRAIN: Yield ALL valid 7-consecutive-season sliding windows
        for i in range(num_windows):
            if start_years[i + 6] == start_years[i] + 6:
                base_year = start_years[i]
                samples.append({
                    'SPLIT': 'Train',
                    'PLAYER_NAME': player_name,
                    'PLAYER_ID': player_id,
                    'INPUT_SEASON': f"{base_year}-{base_year + 4}",
                    'FORECAST_SEASON': f"{base_year + 4}-{base_year + 7}"
                })

# 6. Create the final dataframe with strictly capitalized column names and specific order
final_split_df = pd.DataFrame(samples, columns=['SPLIT', 'PLAYER_NAME', 'PLAYER_ID', 'INPUT_SEASON', 'FORECAST_SEASON'])
final_split_df = final_split_df.sort_values(by=['SPLIT', 'PLAYER_ID', 'INPUT_SEASON']).reset_index(drop=True)

# --- Verification ---
print("\n--- Generated Data Points (Train vs Test) ---")
split_counts = final_split_df['SPLIT'].value_counts()
print(f"Total training data samples generated: {split_counts.get('Train', 0)}")
print(f"Total test data samples generated: {split_counts.get('Test', 0)}")

# Save the final file
final_split_df.to_csv(DATA_DIR / "player_train_test_split.csv", index=False)


--- Generated Data Points (Train vs Test) ---
Total training data samples generated: 2112
Total test data samples generated: 145
